# MARC/Spreadsheet → Wikidata Reconciliation → BIBFRAME RDF / Live API

Interactive notebook version of `bibframe_pipeline.py` and
`push_to_bcld.py`. Run cells top to bottom.

**Files this notebook expects to sit next to** (same folder):
- `bibframe_pipeline.py`
- `push_to_bcld.py`
- your source data (MARC `.mrc`) — this shortened version offers
  eleven sample MARC files (down from the full set of twenty-nine):
  1 general book/media sample, 3 plain-sequential batches, 3 batches
  selected for high identifier/$0/$1 density, and 4 batches filtered
  to academic journals only. See the **Configure** cell below, or
  `bibframe_pipeline.py`'s `SAMPLE_FILES` list for the full set this
  was trimmed from.

Each stage below is its own cell so you can inspect results before
moving to the next one — especially before the live API push at the
end, which writes real data.


## 0. Setup

In [ ]:
# Run once per environment. If your environment reports
# "externally-managed-environment", add --break-system-packages,
# or install into a virtualenv/conda env instead.
%pip install -q requests rapidfuzz rdflib pymarc pandas


In [ ]:
import sys, json
from pathlib import Path

import pandas as pd

# Import the pipeline modules as libraries (their CLI `main()` only
# runs when executed directly, so importing here is safe).
sys.path.insert(0, str(Path.cwd()))
import bibframe_pipeline as bp
import push_to_bcld as api


## 1. Configure

Set your input file and, if you'll be pushing to the live API, the
base URL and an auth token. Use `getpass` for the token so it doesn't
get typed into a cell or saved in the notebook file.


In [ ]:
# Eleven sample MARC files are included (a shortened set -- see
# bibframe_pipeline.py's SAMPLE_FILES list for the full 29):
#   MARC-sample-20260722-2455.mrc          -- 3 real MARC21 book/media records
#                                              (Anne of Green Gables, Animal Farm,
#                                              Data Feminism, Algorithms of Oppression,
#                                              etc.) -- mostly book-style with 100 main
#                                              entries, several with $0 LC NAF authority
#                                              links on the author
#   sample_BIB250403_batch1-25.mrc         -- records 0-24 of BIB250403.mrc
#   sample_BIB250403_batch2-25.mrc         -- records 25-49 of BIB250403.mrc
#   sample_BIB250403_batch3-25.mrc         -- records 50-74 of BIB250403.mrc
#     (these 3 are non-overlapping slices of the first 200 records in
#     BIB250403.mrc -- plain sequential order, richly-cataloged academic/
#     research material)
#   sample_BIB250403_richIDs_batch1-25.mrc -- 3 batches of 25, but these are the
#   sample_BIB250403_richIDs_batch2-25.mrc    HIGHEST-scoring records from across
#   sample_BIB250403_richIDs_batch3-25.mrc    BIB250403.mrc's full 68,100 records,
#     ranked by (identifiers) + (count of $0/$1 subfields). ~70-85% of records in
#     each batch have a populated author_identifiers, vs. ~10-20% in the plain-
#     sequential batches above.
#   sample_BIB250403_journals_batch1-25.mrc -- 4 batches of 25: academic journals
#   sample_BIB250403_journals_batch2-25.mrc    specifically (Leader/07=='s' serial,
#   sample_BIB250403_journals_batch3-25.mrc    008/21=='p' periodical, title contains
#   sample_BIB250403_journals_batch4-25.mrc    "journal"), with e-flux explicitly
#     excluded. 200 records sampled evenly across all 1,449 matching records in
#     BIB250403.mrc for topical diversity across disciplines.
# Uncomment ONE of the lines below.
INPUT_FILE = "sample_BIB250403_journals_batch3-25.mrc"
# INPUT_FILE = "MARC-sample-20260722-2455.mrc"
# INPUT_FILE = "sample_BIB250403_batch1-25.mrc"
# INPUT_FILE = "sample_BIB250403_batch2-25.mrc"
# INPUT_FILE = "sample_BIB250403_batch3-25.mrc"
# INPUT_FILE = "sample_BIB250403_richIDs_batch1-25.mrc"
# INPUT_FILE = "sample_BIB250403_richIDs_batch2-25.mrc"
# INPUT_FILE = "sample_BIB250403_richIDs_batch3-25.mrc"
# INPUT_FILE = "sample_BIB250403_journals_batch1-25.mrc"
# INPUT_FILE = "sample_BIB250403_journals_batch2-25.mrc"
# INPUT_FILE = "sample_BIB250403_journals_batch4-25.mrc"

RDF_OUT_PATH = "catalog.ttl"
RDF_FORMAT = "turtle"              # turtle | json-ld | xml | nt
REVIEW_OUT_PATH = "needs_review.csv"

API_BASE_URL = "https://dev.bcld.info/api"


In [ ]:
from getpass import getpass

# This deployment authenticates via Keycloak (realm "bluecore" at
# https://dev.bcld.info/keycloak/), not a static API token. You need:
#   - your Keycloak username/password
#   - the application's OAuth client_id (NOT Keycloak's built-in
#     "account"/"account-console" clients). Two app-specific clients
#     exist in this realm: "bluecore_api" and "bluecore_workflows".
#     "bluecore_api" is the default guess below since it should map
#     to the /works/, /instances/ endpoints this script calls;
#     "bluecore_workflows" more likely covers the /batches/ and
#     /export/ (Airflow-triggered) endpoints instead.
#   - a client_secret, only if that client is confidential

KEYCLOAK_CLIENT_ID = "bluecore_api"  # best-guess default -- try "bluecore_workflows"
                                      # instead if this one gets rejected
KEYCLOAK_CLIENT_SECRET = ""          # leave blank if the client is public (no secret)

# Leave the username blank and press Enter on both prompts to skip
# auth entirely (e.g. if you're only building the RDF file in this
# session and not pushing to the API).
keycloak_username = input("Keycloak username (leave blank to skip): ")
API_TOKEN = None

if keycloak_username:
    keycloak_password = getpass("Keycloak password: ")
    if not KEYCLOAK_CLIENT_ID:
        raise ValueError("Set KEYCLOAK_CLIENT_ID above before logging in.")
    API_TOKEN = api.get_keycloak_token(
        username=keycloak_username,
        password=keycloak_password,
        client_id=KEYCLOAK_CLIENT_ID,
        client_secret=KEYCLOAK_CLIENT_SECRET or None,
    )
    print("Keycloak login OK -- access token obtained.")
else:
    print("Skipping auth -- API_TOKEN is None.")


## 2. Load records

In [ ]:
records = bp.load_records(INPUT_FILE)
print(f"Loaded {len(records)} records from {INPUT_FILE}")

pd.DataFrame([{
    "local_id": r.local_id, "title": r.title, "author": r.author,
    "pub_date": r.pub_date, **r.identifiers,
} for r in records])


## 3. Reconcile against Wikidata

Identifier matches (ISBN/ISSN/OCLC/LCCN/DOI/VIAF) auto-accept. Fuzzy
title-only matches are always routed to review — see
`bibframe_pipeline.py`'s module docstring for the reasoning.

This makes live calls to the Wikidata SPARQL endpoint and
`wbsearchentities` API, so it needs outbound network access and will
take a few seconds per record.


In [ ]:
for i, r in enumerate(records, 1):
    bp.find_candidates(r)
    bp.score_candidates(r)
    best = f"{r.best.qid} ({r.best.entity_role})" if r.best else "-"
    print(f"[{i}/{len(records)}] {r.title[:50]!r:52} -> {r.decision:12} {best}")

pd.Series([r.decision for r in records]).value_counts()


## 4. Inspect results before doing anything with them

Anything that isn't `auto_accept` is worth a look before you build
the RDF or push it live — this is the same information that would
otherwise only show up in `needs_review.csv`.

**This step is read-only and doesn't filter anything out.** Every
record — `auto_accept`, `needs_review`, or `no_match` alike — still
gets built into the RDF and still gets pushed in steps 5a/5b below;
nothing here removes a record from that. The decision only controls
whether a Wikidata link gets attached to that record:
- `auto_accept` → gets an `rdfs:seeAlso` link to the matched Wikidata item
- `needs_review` → gets an `rdfs:comment` flagging the candidate, but no link
- `no_match` → gets neither, just its own bibliographic data

If you want to actually exclude some records (e.g. only push the ones
that auto-accepted), you'd filter the `records` list yourself before
step 5a/5b — that's not done automatically anywhere in this notebook.


In [ ]:
review_rows = [{
    "local_id": r.local_id, "title": r.title, "decision": r.decision,
    "best_qid": r.best.qid if r.best else None,
    "score": r.best.score if r.best else None,
    "entity_role": r.best.entity_role if r.best else None,
} for r in records if r.decision != "auto_accept"]

print(f"{len(review_rows)}/{len(records)} record(s) are not auto_accept -- "
      f"they'll still be built/pushed below, just without a confirmed Wikidata link.")
pd.DataFrame(review_rows) if review_rows else "Nothing needs review -- every record auto-accepted."


## 5a. Build the BIBFRAME RDF file

Writes a portable RDF file (Turtle/JSON-LD/RDF-XML/N-Triples) — nothing
gets sent to any server in this step.


In [ ]:
graph = bp.build_bibframe_graph(records)
bp.serialize_graph(graph, RDF_OUT_PATH, RDF_FORMAT)
bp.write_review_csv(records, REVIEW_OUT_PATH)

print(f"Wrote {len(graph)} triples to {RDF_OUT_PATH}")
print(f"Review queue: {REVIEW_OUT_PATH}")


In [ ]:
# Preview the first part of the serialized RDF
print(Path(RDF_OUT_PATH).read_text()[:2000])


## 5b. (Optional) Push directly to the live API

**Writes real data.** Run the dry-run cell first and read the printed
payloads — nothing is sent to the network in that cell. Only run the
live cell after that looks right, and after you've confirmed the auth
scheme matches what `dev.bcld.info` actually expects (this notebook
defaults to `Authorization: Bearer <token>`, see `push_to_bcld.py`'s
docstring).


In [ ]:
# --- Dry run: prints every payload, makes NO network calls ---
client = api.BcldApiClient(API_BASE_URL, token=API_TOKEN)
dry_rows = api.push_records(records, client, dry_run=True)


In [ ]:
# --- Live push: only run this after reviewing the dry run above ---
CONFIRM_LIVE_PUSH = False  # flip to True deliberately before running

if CONFIRM_LIVE_PUSH:
    if not API_TOKEN:
        raise RuntimeError("No API_TOKEN set — re-run the getpass cell above first.")
    live_rows = api.push_records(records, client, dry_run=False)
    api.write_manifest(live_rows, "push_manifest.csv")
    display(pd.DataFrame(live_rows))
else:
    print("CONFIRM_LIVE_PUSH is False — nothing was sent. "
          "Set it to True in this cell and re-run to actually push.")


## Notes

- Re-running the reconciliation cell re-queries Wikidata for every
  record; consider caching `records` (e.g. `pickle`) if you're
  iterating on the RDF/push steps and don't want to re-hit the API
  each time.
- The live push (5b) is **not idempotent** — re-running it against
  the same records creates duplicate Works/Instances. Check
  `push_manifest.csv` before re-running.
- Full field mappings, MARC tag reference, and design rationale are in
  `README.md` alongside these scripts.
